# Sisyphus Watch

When public stories change, summaries can lie.

Case:
NASA/Boeing Starliner Crew Flight Test

A crewed spacecraft test began with an expected Starliner return path, then shifted after technical issues and uncertainty led NASA to return Starliner without crew.


Setup note: this cell checks that the attached Kaggle dataset matches this notebook version.


In [ ]:
from pathlib import Path
import json
import importlib.util
import sys

try:
    from IPython.display import FileLink, HTML, Markdown, display
except ModuleNotFoundError:
    class HTML(str):
        pass

    class Markdown(str):
        pass

    FileLink = None

    def display(obj):
        text = str(obj)
        print(text[:1200] + ("..." if len(text) > 1200 else ""))


def _candidate_project_roots(start=None):
    seen = set()

    def add(path: Path):
        resolved = path.expanduser().resolve()
        if resolved.is_file():
            resolved = resolved.parent
        for candidate in [resolved, *resolved.parents]:
            if candidate not in seen:
                seen.add(candidate)
                yield candidate

    if start is not None:
        yield from add(start)

    yield from add(Path.cwd())

    kaggle_working = Path("/kaggle/working")
    if kaggle_working.exists() and kaggle_working not in seen:
        seen.add(kaggle_working)
        yield kaggle_working

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for module_path in kaggle_input.glob("**/src/sisyphus_watch_demo.py"):
            root = module_path.parents[1].resolve()
            if root not in seen:
                seen.add(root)
                yield root


def _load_sisyphus_demo_module():
    for root in _candidate_project_roots(Path.cwd()):
        module_path = root / "src" / "sisyphus_watch_demo.py"
        if module_path.exists():
            src_path = str(root / "src")
            if src_path not in sys.path:
                sys.path.insert(0, src_path)
            spec = importlib.util.spec_from_file_location("sisyphus_watch_demo", module_path)
            if spec is None or spec.loader is None:
                continue
            module = importlib.util.module_from_spec(spec)
            sys.modules["sisyphus_watch_demo"] = module
            spec.loader.exec_module(module)
            return module

    raise FileNotFoundError(
        "Could not locate src/sisyphus_watch_demo.py. Expected a repo root, "
        "notebook inside notebooks/, or a Kaggle input dataset containing the project folder."
    )


demo = _load_sisyphus_demo_module()
import sisyphus_watch_demo as swd

print("Loaded Sisyphus module:", swd.__file__)

_REQUIRED_SISYPHUS_SYMBOLS = [
    "render_product_brief_html",
    "render_review_map_html",
    "build_surface_model",
    "render_agent_contact_surface_html",
    "render_discovery_packet_html",
    "render_guided_flow_html",
    "render_case_hook_html",
    "render_what_changed_html",
]
_missing_sisyphus_symbols = [
    name for name in _REQUIRED_SISYPHUS_SYMBOLS if not hasattr(swd, name)
]
if _missing_sisyphus_symbols:
    raise RuntimeError(
        "The attached Kaggle dataset contains an older src/sisyphus_watch_demo.py than this notebook expects. "
        "Re-upload or reattach a Kaggle dataset built from the same GitHub commit as this notebook. "
        f"Resolved module path: {swd.__file__}. "
        f"Missing symbols: {', '.join(_missing_sisyphus_symbols)}."
    )

PROJECT_ROOT = swd.find_project_root(Path.cwd())
src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from sisyphus_watch_demo import (
    build_deterministic_discovery_packet,
    build_guided_flow_summary,
    build_surface_model,
    build_user_problem_packet,
    build_agent_packet,
    fallback_to_demo_records,
    filter_sources_for_card,
    find_project_root,
    get_news_cards,
    get_evidence_patch_for_scenario,
    load_demo_sources,
    load_evidence_patches,
    load_scenario_authoring_template,
    maybe_run_live_extraction,
    render_agent_capability_strip_html,
    render_agent_contact_surface_html,
    render_case_hook_html,
    render_user_problem_card_html,
    render_plain_summary_vs_sisyphus_html,
    render_what_changed_html,
    render_product_brief_html,
    render_guided_flow_html,
    render_discovery_packet_html,
    render_judge_quickstart_html,
    render_run_status_html,
    render_review_map_html,
    render_course_concepts_html,
    render_export_artifacts_overview_html,
    render_submission_readiness_html,
    render_surface_model_html,
    maybe_run_google_ai_discovery,
    render_branch_view_html,
    render_agent_workflow_trace_html,
    render_kaggle_midcheck_summary_html,
    render_reviewer_dashboard_html,
    render_card_html,
    render_json_export,
    render_epistemic_layers_html,
    render_evaluation_summary_html,
    render_quality_checks_html,
    render_sources_table_html,
    render_version_timeline_html,
    render_claim_drift_html,
    render_claim_graph_html,
    render_graph_query_preview_html,
    render_reviewer_presets_html,
    render_revision_proposal_html,
    render_revision_comparison_html,
    render_scenario_authoring_preview_html,
    run_quality_checks,
    select_news_card,
    to_all_cards_jsonl,
    to_jsonl,
    write_export_artifacts,
)

from sisyphus_watch_adk_demo import (
    build_adk_capability_manifest,
    run_sisyphus_adk_demo,
)
from sisyphus_watch_mcp_server import build_mcp_capability_manifest

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
USER_PROBLEM = "How did the public story around Boeing Starliner Crew Flight Test shift from an expected crewed Starliner return to NASA's decision to return Starliner without crew and use a different crew return path?"
SCENARIO_ID = "starliner_crew_return_decision"
# Other deterministic synthetic scenarios remain available for schema and workflow testing.
RUN_GOOGLE_DISCOVERY = False
RUN_LIVE_MODE = False

source_records = load_demo_sources()

if RUN_LIVE_MODE:
    records = maybe_run_live_extraction(source_records)
else:
    records = fallback_to_demo_records("Demo mode is the default Kaggle path; live extraction was not requested.")

all_news_cards = get_news_cards(records)
news_card = select_news_card(records, SCENARIO_ID)
selected_source_records = filter_sources_for_card(source_records, news_card)
agent_packet = build_agent_packet(news_card)
evidence_patches = load_evidence_patches()
evidence_patch = get_evidence_patch_for_scenario(evidence_patches, news_card["scenario_id"])

problem_packet = build_user_problem_packet(
    USER_PROBLEM,
    SCENARIO_ID,
    "google_ai_discovery" if RUN_GOOGLE_DISCOVERY else "deterministic_fixture_discovery",
)

if RUN_GOOGLE_DISCOVERY:
    discovery_packet = maybe_run_google_ai_discovery(USER_PROBLEM, selected_source_records, SCENARIO_ID)
else:
    discovery_packet = build_deterministic_discovery_packet(USER_PROBLEM, selected_source_records, SCENARIO_ID)

# The Starliner case is a frozen deterministic real-case snapshot for reproducible judging.
# Optional Google AI discovery can refresh candidate sources, but it does not change this canonical snapshot.
guided_flow_summary = build_guided_flow_summary(
    news_card,
    selected_source_records,
    discovery_packet=discovery_packet,
    evidence_patch=evidence_patch,
)

adk_manifest = build_adk_capability_manifest()
adk_demo_trace = run_sisyphus_adk_demo(SCENARIO_ID, USER_PROBLEM)
mcp_manifest = build_mcp_capability_manifest()

fallback_reasons = []
if RUN_LIVE_MODE and records.get("fallback_reason"):
    fallback_reasons.append("Record path: " + records["fallback_reason"])
if discovery_packet.get("fallback_reason"):
    fallback_reasons.append("Discovery path: " + discovery_packet["fallback_reason"])

run_status = {
    "run_google_discovery": RUN_GOOGLE_DISCOVERY,
    "run_live_mode": RUN_LIVE_MODE,
    "discovery_mode": discovery_packet.get("mode"),
    "record_mode": records.get("mode", "demo"),
    "fallback_reasons": fallback_reasons,
    "selected_card_id": news_card.get("card_id"),
    "selected_scenario_id": news_card.get("scenario_id", SCENARIO_ID),
    "available_demo_card_count": len(all_news_cards),
    "evidence_patch_available": evidence_patch is not None,
    "export_path_target": "/kaggle/working",
}

surface_model = build_surface_model(
    news_card,
    evidence_patch=evidence_patch,
    discovery_packet=discovery_packet,
    adk_manifest=adk_manifest,
    mcp_manifest=mcp_manifest,
)

display(HTML(render_case_hook_html(news_card, discovery_packet, surface_model)))


## Plain Summary vs Claim-Version Control

A plain summary compresses the case; Sisyphus preserves the changing claim structure.


In [ ]:
display(HTML(render_plain_summary_vs_sisyphus_html(news_card, discovery_packet)))


## What Changed?

The story shifted from expected crewed Starliner return to uncrewed Starliner return decision and a different crew return path.


In [ ]:
display(HTML(render_what_changed_html(news_card)))


## Sisyphus Separation

The agent separates source-bound findings, actor claims, actions, interpretations, and current judgment.


In [ ]:
display(HTML(render_epistemic_layers_html(news_card)))


## Claim Timeline

The timeline shows how the public claim changed as evidence and uncertainty accumulated.


In [ ]:
display(HTML(render_version_timeline_html(news_card)))


## Claim Drift

Drift marks whether claims weakened, narrowed, became complicated, or were revised.


In [ ]:
display(HTML(render_claim_drift_html(news_card)))


## Claim Graph

The graph connects sources, findings, claims, drift, and judgment for downstream reuse.


In [ ]:
display(HTML(render_claim_graph_html(news_card)))


## Evidence Patch / Revision Preview

New evidence is reviewed as a non-mutating patch before any canonical update.


In [ ]:
display(HTML(render_revision_proposal_html(news_card, evidence_patch)))
display(HTML(render_revision_comparison_html(news_card, evidence_patch)))


## Agent Contact Surface

Rendered HTML is for humans; JSON/JSONL/MCP packets are for downstream agents.


In [ ]:
display(HTML(render_agent_contact_surface_html(surface_model, news_card, evidence_patch)))


In [ ]:
kaggle_output_dir = Path("/kaggle/working")
display(HTML(render_export_artifacts_overview_html(news_card, evidence_patch, "/kaggle/working")))

if kaggle_output_dir.exists():
    export_paths = write_export_artifacts(news_card, kaggle_output_dir)
    display(Markdown("Export artifacts written to `/kaggle/working`."))
    if FileLink is not None:
        for label, export_path in export_paths.items():
            display(FileLink(str(export_path), result_html_prefix=f"{label}: "))
    else:
        for label, export_path in export_paths.items():
            print(f"{label}: {export_path}")
else:
    display(Markdown("Local run: `/kaggle/working` is unavailable, so export artifact writing was skipped."))


## Course Concepts / Technical Appendix

The course-concept mapping is kept compact and visible for the rubric.


In [ ]:
display(HTML(render_course_concepts_html(adk_manifest, adk_demo_trace, mcp_manifest)))
display(HTML(render_review_map_html(
    surface_model,
    run_status=run_status,
    adk_manifest=adk_manifest,
    mcp_manifest=mcp_manifest,
)))
display(HTML(render_surface_model_html(surface_model)))
display(HTML(render_discovery_packet_html(discovery_packet)))
display(HTML(render_guided_flow_html(guided_flow_summary)))
display(HTML(render_run_status_html(run_status)))


## Submission Readiness

The final check confirms that the deterministic Starliner story demo, human workflow, agent contact surface, and export artifacts are ready for review.


In [ ]:
checks = run_quality_checks(news_card)
display(HTML(render_submission_readiness_html(
    news_card,
    evidence_patch,
    checks,
    discovery_packet=discovery_packet,
    adk_manifest=adk_manifest,
    mcp_manifest=mcp_manifest,
)))
display(HTML(render_evaluation_summary_html(checks, news_card)))
display(HTML(render_quality_checks_html(checks)))
display(HTML(render_kaggle_midcheck_summary_html(news_card, evidence_patch)))

if not all(row["status"] == "PASS" for row in checks):
    failed = [row for row in checks if row["status"] != "PASS"]
    raise AssertionError(f"Sisyphus Watch demo checks failed: {failed}")

print("Demo mode PASS: deterministic Starliner story, human workflow, agent contact surface, and export checks are available.")


## Reproducibility / Dataset Version Sync

The default path is deterministic, no-key, and no-network. If the setup cell fails, re-upload the matching Kaggle dataset version and restart the kernel before running the notebook again. Optional Google AI discovery is review-only and does not mutate the canonical Starliner snapshot.


## Limitations

- Sisyphus Watch is not an automatic truth oracle.
- The Starliner scenario is a frozen public-source snapshot, not live verification.
- It does not assign blame, make legal conclusions, or issue a final engineering-certification judgment.
- Optional Google AI discovery candidates are review-only and do not mutate the canonical card.
- Synthetic scenarios remain available as secondary schema and workflow tests.
- Generated image prompts are not evidence.
- The demo does not implement live web ingestion, crawling, accounts, recommendations, a database, or a production news platform.
